# Daily Sales Forecasting

## Business Context

This dataset contains daily sales records from a retail company. The goal is to understand historical sales patterns and forecast future sales to support inventory planning, staffing, and cash flow decisions.

## Objective

Build a reliable time series model that forecasts daily sales for the next 14 days.

## Approach

The analysis follows these steps:
- Data loading and cleaning
- Exploratory analysis and visualization
- Stationarity testing and transformations
- Model selection (ARIMA)
- Forecast evaluation and future predictions

In [1]:
# Import libraries we'll need
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

In [2]:
# -------------------------------------------------
# SECTION 2: Load and explore the data
# -------------------------------------------------

# Load the CSV file
df = pd.read_csv("dailysales.csv")

# Parse dates (format: 01-Jan-18)
df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y')

# Sort by date just in case
df = df.sort_values('date').reset_index(drop=True)

# Take a look at what we have
print("First few rows:")
print(df.head())
print("\nData info:")
print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())

First few rows:
        date  sales
0 2018-01-01  477.0
1 2018-01-02  365.0
2 2018-01-03  442.0
3 2018-01-04  490.0
4 2018-01-05  396.0

Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    704 non-null    datetime64[ns]
 1   sales   704 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 11.1 KB
None

Missing values per column:
date     0
sales    0
dtype: int64


In [3]:
# -------------------------------------------------
# SECTION 3: Clean up duplicates and create time series
# -------------------------------------------------

# Sort by date and remove duplicate dates (keep the last value)
df = df.sort_values('date').drop_duplicates(subset=['date'], keep='last')

# Create the time series
ts = df.set_index('date')['sales']

print(f"Total records: {len(ts)}")
print(f"Date range: {ts.index.min()} to {ts.index.max()}")
print(f"Missing dates: {ts.index.max() - ts.index.min() - pd.Timedelta(days=len(ts)-1)}")

# Check for any weird values
print(f"\nSales summary:")
print(ts.describe())

Total records: 703
Date range: 2018-01-01 00:00:00 to 2019-12-31 00:00:00
Missing dates: 27 days 00:00:00

Sales summary:
count     703.000000
mean      261.483499
std       272.346053
min        35.900000
25%       151.975000
50%       214.600000
75%       292.500000
max      4000.000000
Name: sales, dtype: float64


In [4]:
# Check for missing dates in the time series
full_index = pd.date_range(ts.index.min(), ts.index.max(), freq="D")
missing_dates = full_index.difference(ts.index)

print(f"Expected dates: {len(full_index)}")
print(f"Actual dates: {len(ts)}")
print(f"Missing dates: {len(missing_dates)}")

if len(missing_dates) > 0:
    print("\nMissing dates found:")
    print(missing_dates[:20])  # Show first 20 missing dates
    if len(missing_dates) > 20:
        print(f"... and {len(missing_dates) - 20} more")

Expected dates: 730
Actual dates: 703
Missing dates: 27

Missing dates found:
DatetimeIndex(['2018-02-17', '2018-02-18', '2018-08-01', '2018-08-06',
               '2018-08-11', '2018-08-12', '2018-08-13', '2018-11-03',
               '2018-12-02', '2018-12-31', '2019-01-29', '2019-02-17',
               '2019-02-18', '2019-03-01', '2019-03-30', '2019-04-28',
               '2019-05-27', '2019-06-25', '2019-07-24', '2019-08-11'],
              dtype='datetime64[ns]', freq=None)
... and 7 more


In [5]:
# Fill in missing dates with NaN so we have a continuous time series
ts = ts.reindex(full_index)

# Count how many NaN values we have (these are the missing dates)
missing_count = ts.isna().sum()
print(f"Number of missing daily sales values: {missing_count}")

# Take a look at the data now
print("\nFirst few rows with missing dates:")
print(ts.head(20))

Number of missing daily sales values: 27

First few rows with missing dates:
2018-01-01    477.0
2018-01-02    365.0
2018-01-03    442.0
2018-01-04    490.0
2018-01-05    396.0
2018-01-06    385.0
2018-01-07    492.0
2018-01-08    331.0
2018-01-09    249.0
2018-01-10    258.0
2018-01-11    358.0
2018-01-12    230.0
2018-01-13    103.0
2018-01-14    222.0
2018-01-15    262.0
2018-01-16    245.0
2018-01-17    204.0
2018-01-18    263.0
2018-01-19    251.0
2018-01-20    131.0
Freq: D, Name: sales, dtype: float64


In [6]:
# Count and show where the gaps are
missing_count = ts.isna().sum()
print(f"Total missing days: {missing_count}")

# Find the indices where data is missing
missing_indices = ts[ts.isna()].index
print(f"\nFirst 10 missing dates:")
print(missing_indices[:10])

# For now, let's fill missing values using forward fill
# (using the last known value to fill the gap)
ts_filled = ts.fillna(method='ffill')

# Check if we still have any NaN values at the beginning
if ts_filled.isna().any():
    # If first values are NaN, use backward fill
    ts_filled = ts_filled.fillna(method='bfill')

print(f"\nAfter filling: {ts_filled.isna().sum()} missing values remain")

Total missing days: 27

First 10 missing dates:
DatetimeIndex(['2018-02-17', '2018-02-18', '2018-08-01', '2018-08-06',
               '2018-08-11', '2018-08-12', '2018-08-13', '2018-11-03',
               '2018-12-02', '2018-12-31'],
              dtype='datetime64[ns]', freq=None)

After filling: 0 missing values remain


In [7]:
# Fill missing dates using linear interpolation
# This estimates missing values based on surrounding dates

print(f"Data range: {ts.index.min().date()} to {ts.index.max().date()}")
print(f"Total days in range: {len(ts)}")
print(f"Missing days found: {ts.isna().sum()}")

# Fill missing values using linear interpolation
ts = ts.interpolate(method='linear')

# Double-check nothing is still missing
if ts.isna().sum() == 0:
    print("\nAll missing values have been filled.")
else:
    print(f"\nWarning: {ts.isna().sum()} values still missing.")

Data range: 2018-01-01 to 2019-12-31
Total days in range: 730
Missing days found: 27

All missing values have been filled.
